In [ ]:
import cv2
import numpy as np

# load calibration
data = np.load('../pi5-camera-tool/stereo.npy', allow_pickle=True).item()
Kl, Dl, Kr, Dr, R, T, img_size = data['Kl'], data['Dl'], data['Kr'], data['Dr'], \
                                 data['R'], data['T'], data['img_size']
R1, R2, P1, P2, Q, validRoi1, validRoi2 = cv2.stereoRectify(Kl, Dl, Kr, Dr, img_size, R, T)
xmap1, ymap1 = cv2.initUndistortRectifyMap(Kl, Dl, R1, P1, img_size, cv2.CV_32FC1)
xmap2, ymap2 = cv2.initUndistortRectifyMap(Kr, Dr, R2, P2, img_size, cv2.CV_32FC1)

# 1. Load left and right images (rectified)
left_image = cv2.imread('./test-data/left-0.png', cv2.IMREAD_GRAYSCALE)
right_image = cv2.imread('./test-data/right-0.png', cv2.IMREAD_GRAYSCALE)

# apply calibration
left_image = cv2.remap(left_image, xmap2, ymap2, cv2.INTER_LINEAR)
right_image = cv2.remap(right_image, xmap2, ymap2, cv2.INTER_LINEAR)

# 2. Create Stereo Matcher (StereoBM or StereoSGBM)
max_disp = 128
wsize = 31
left_matcher = cv2.StereoSGBM_create(
    minDisparity=0,
    numDisparities=max_disp,
    blockSize=16,
    P1=8 * 3 * 5**2,
    P2=32 * 3 * 5**2,
    disp12MaxDiff=1,
    uniquenessRatio=15,
    speckleWindowSize=0,
    speckleRange=2,
    preFilterCap=63,
    mode=cv2.STEREO_SGBM_MODE_SGBM_3WAY
)
right_matcher = cv2.ximgproc.createRightMatcher(left_matcher)

# 3. Compute raw disparity maps
left_disp = left_matcher.compute(left_image, right_image).astype(np.float32) / 16.0
right_disp = right_matcher.compute(right_image, left_image).astype(np.float32) / 16.0

# 4. Create WLS Filter
wls_filter = cv2.ximgproc.createDisparityWLSFilter(left_matcher)
wls_filter.setLambda(8000.0)
wls_filter.setSigmaColor(1.5)

# 5. Filter the disparity map
filtered_disp = wls_filter.filter(left_disp, left_image, disparity_map_right=right_disp)

# 6. Visualize results
cv2.imshow('Raw Disparity', (left_disp * 16).astype(np.uint8))
cv2.imshow('Filtered Disparity', (filtered_disp * 16).astype(np.uint8))
cv2.waitKey()